# 04 - OKX US Connector Exploration

**Goal**: Explore the OKX US public REST API (V5) to inform our 4th production connector.

**Background**: OKCoin ceased all trading operations May 15, 2025. OKX US is the
legal successor entity (formerly OKCoin USA Inc.), launched April 2025. Ohio MTL
confirmed via Ohio Division of Financial Institutions. Connector name: `okx`.

**Scope**:
- Determine working base URL for OKX US API
- Test raw `httpx` approach (custom connector per DEC-005/DEC-011)
- Map OKX symbols to canonical pairs
- Parse responses into our `TopOfBook` dataclass
- Compare batch ticker vs per-pair ticker efficiency
- Document rate limits, error handling, edge cases
- Answer key design questions for the production connector

**Supported Pairs** (from PROJECT_INSTRUCTIONS.md):
- BTC/USD, BTC/USDC
- LTC/USD, LTC/USDC, LTC/BTC
- SOL/USD, SOL/USDC, SOL/BTC

**Key Differences from Kraken, Coinbase & Gemini**:
- OKX uses dash-separated uppercase symbols: `BTC-USD` (same as Coinbase)
- V5 API envelope: `{"code": "0", "msg": "", "data": [...]}`
- Ticker endpoint includes bid/ask prices AND sizes (unlike Gemini)
- Batch ticker endpoint: all SPOT pairs in 1 call (like Kraken)
- Timestamps are Unix milliseconds as strings (no conversion needed)
- No existing Python SDK needed — raw httpx (consistent with all connectors)

**Lessons Applied** (from LESSONS_LEARNED.md):
- LL-001: Verify exact symbol format, don't assume
- LL-002: Document actual response shapes from live API, not just docs
- LL-003: Test rate limit behavior before building production connector
- LL-010: All prices/sizes via `to_decimal()`, never float
- LL-050: Use `nest_asyncio.apply()` for async in Jupyter
- LL-052: Don't assume batch endpoints exist — verify
- LL-062: Always verify chosen endpoint provides ALL fields needed for TopOfBook
- LL-065: Import from `venues.symbol_translator` (NOT `venues.symbols`)

## 1. Setup & Base URL Discovery

In [1]:
# Install dependencies (run once)
# !pip install httpx nest_asyncio

In [2]:
import json
import sys
import time
from pprint import pprint

import httpx
import nest_asyncio

# Enable nested event loops for Jupyter (LL-050)
nest_asyncio.apply()

sys.path.insert(0, "../src")

# Our existing infrastructure — reuse, don't reimplement
# Import path is venues.symbol_translator (NOT venues.symbols — LL-065)
from uscryptoarb.marketdata.topofbook import TopOfBook, tob_from_raw
from uscryptoarb.venues.symbol_translator import SymbolTranslator

print("Imports complete.")

Imports complete.


In [3]:
# OKX US may use different base URLs depending on the regional entity.
# OKX documentation references:
#   - app.okx.com    (OKX US regional portal)
#   - www.okx.com    (OKX Global)
#   - www.okcoin.com (legacy OKCoin — may still route to V5 API)
#
# Test each candidate and pick the one that works for US market data.

CANDIDATE_URLS = [
    "https://www.okx.com",
    "https://app.okx.com",
    "https://www.okcoin.com",
]

TEST_ENDPOINT = "/api/v5/market/ticker?instId=BTC-USD"

print(f"{'Base URL':<30s} {'Status':>6s}  {'Code':>5s}  {'instId':>10s}  {'bidPx':>12s}")
print("-" * 80)

working_urls = []
for base in CANDIDATE_URLS:
    try:
        resp = httpx.get(base + TEST_ENDPOINT, timeout=10.0)
        body = resp.json()
        code = body.get("code", "?")
        data = body.get("data", [{}])
        inst_id = data[0].get("instId", "?") if data else "?"
        bid_px = data[0].get("bidPx", "?") if data else "?"
        status = resp.status_code
        print(f"{base:<30s} {status:>6d}  {code:>5s}  {inst_id:>10s}  {bid_px:>12s}")
        if code == "0" and data:
            working_urls.append(base)
    except Exception as exc:
        print(f"{base:<30s}  ERROR: {exc}")

print(f"\nWorking URLs: {working_urls}")

Base URL                       Status   Code      instId         bidPx
--------------------------------------------------------------------------------
https://www.okx.com               200      0     BTC-USD       68622.9
https://app.okx.com               200      0     BTC-USD       68622.9
https://www.okcoin.com          ERROR: Expecting value: line 1 column 1 (char 0)

Working URLs: ['https://www.okx.com', 'https://app.okx.com']


In [4]:
# Pick the best working URL.
# Prefer app.okx.com (US regional) if it works, otherwise www.okx.com.
if "https://app.okx.com" in working_urls:
    BASE_URL = "https://app.okx.com"
elif "https://www.okx.com" in working_urls:
    BASE_URL = "https://www.okx.com"
elif working_urls:
    BASE_URL = working_urls[0]
else:
    raise RuntimeError("No working OKX API base URL found!")

print(f"Using BASE_URL = {BASE_URL}")

Using BASE_URL = https://app.okx.com


## 2. Symbol Discovery & Mapping

OKX uses dash-separated uppercase symbols: `BTC-USD`, `LTC-BTC`, `SOL-USDC`.
This is the same format as Coinbase.

OKX US is a limited subset of OKX Global (~150 assets vs 700+).
We need to verify which of our 8 target pairs actually exist.

In [5]:
# Fetch ALL spot tickers — this doubles as pair discovery
resp = httpx.get(f"{BASE_URL}/api/v5/market/tickers", params={"instType": "SPOT"})
body = resp.json()
print(f"Response code: {body['code']}")
print(f"Total SPOT instruments: {len(body['data'])}")

# Extract all instrument IDs
all_inst_ids = sorted([d["instId"] for d in body["data"]])
print(f"\nFirst 30 instruments: {all_inst_ids[:30]}")

Response code: 0
Total SPOT instruments: 736

First 30 instruments: ['1INCH-EUR', '1INCH-USD', '1INCH-USDT', '2Z-USD', '2Z-USDT', 'A-USD', 'A-USDT', 'AAVE-EUR', 'AAVE-USD', 'AAVE-USDT', 'ACE-USD', 'ACE-USDT', 'ACH-USD', 'ACH-USDT', 'ACT-TRY', 'ACT-USDT', 'ADA-EUR', 'ADA-TRY', 'ADA-USD', 'ADA-USDC', 'ADA-USDT', 'AERGO-USD', 'AERGO-USDT', 'AEVO-USD', 'AEVO-USDT', 'AGLD-USD', 'AGLD-USDT', 'AIXBT-USD', 'AIXBT-USDT', 'ALGO-EUR']


In [6]:
# Check which of our 8 target pairs exist on OKX US
# Expected OKX format: BTC-USD (dash-separated, uppercase)
TARGET_PAIRS = [
    "BTC-USD",
    "BTC-USDC",
    "LTC-USD",
    "LTC-USDC",
    "LTC-BTC",
    "SOL-USD",
    "SOL-USDC",
    "SOL-BTC",
]

inst_id_set = set(all_inst_ids)

print(f"{'Target':12s} {'Available':>10s}")
print("-" * 25)
found_pairs = {}
missing_pairs = []
for target in TARGET_PAIRS:
    exists = target in inst_id_set
    status = "YES" if exists else "NO"
    print(f"{target:12s} {status:>10s}")
    if exists:
        found_pairs[target] = target
    else:
        missing_pairs.append(target)

print(f"\nFound: {len(found_pairs)}/8 target pairs")
if missing_pairs:
    print(f"MISSING: {missing_pairs}")
    print("\n⚠️  Missing pairs will limit arbitrage comparisons for those markets.")
    print("    The production connector should only include available pairs.")

Target        Available
-------------------------
BTC-USD             YES
BTC-USDC            YES
LTC-USD             YES
LTC-USDC            YES
LTC-BTC              NO
SOL-USD             YES
SOL-USDC            YES
SOL-BTC             YES

Found: 7/8 target pairs
MISSING: ['LTC-BTC']

⚠️  Missing pairs will limit arbitrage comparisons for those markets.
    The production connector should only include available pairs.


In [7]:
# Also check if OKX US has USDT pairs (common on OKX Global, less so on US)
# and any other BTC/LTC/SOL pairs we might want in the future
related = [
    s for s in all_inst_ids if any(s.startswith(prefix) for prefix in ["BTC-", "LTC-", "SOL-"])
]
print(f"All BTC/LTC/SOL pairs on OKX US ({len(related)}):")
for s in related:
    marker = " ← TARGET" if s in found_pairs else ""
    print(f"  {s}{marker}")

All BTC/LTC/SOL pairs on OKX US (21):
  BTC-AED
  BTC-AUD
  BTC-BRL
  BTC-EUR
  BTC-TRY
  BTC-USD ← TARGET
  BTC-USDC ← TARGET
  BTC-USDT
  LTC-EUR
  LTC-USD ← TARGET
  LTC-USDC ← TARGET
  LTC-USDT
  SOL-AED
  SOL-AUD
  SOL-BRL
  SOL-BTC ← TARGET
  SOL-EUR
  SOL-TRY
  SOL-USD ← TARGET
  SOL-USDC ← TARGET
  SOL-USDT


In [8]:
# Build the canonical → OKX symbol map (only available pairs)
# Canonical format: "BTC/USD" → OKX format: "BTC-USD"
OKX_SYMBOL_MAP = {}
for okx_sym in found_pairs:
    # Convert BTC-USD → BTC/USD
    canonical = okx_sym.replace("-", "/")
    OKX_SYMBOL_MAP[canonical] = okx_sym

print("OKX_SYMBOL_MAP:")
for k, v in sorted(OKX_SYMBOL_MAP.items()):
    print(f"  {k!r:14s} → {v!r}")

# Create SymbolTranslator and verify round-trip
okx_translator = SymbolTranslator(
    venue="okx",
    canonical_to_venue=OKX_SYMBOL_MAP,
)

print("\nRound-trip verification:")
for canonical in sorted(OKX_SYMBOL_MAP.keys()):
    venue_sym = okx_translator.to_venue_symbol(canonical)
    back = okx_translator.to_canonical(venue_sym)
    match = "✓" if back == canonical else "✗"
    print(f"  {canonical} → {venue_sym} → {back}  {match}")

OKX_SYMBOL_MAP:
  'BTC/USD'      → 'BTC-USD'
  'BTC/USDC'     → 'BTC-USDC'
  'LTC/USD'      → 'LTC-USD'
  'LTC/USDC'     → 'LTC-USDC'
  'SOL/BTC'      → 'SOL-BTC'
  'SOL/USD'      → 'SOL-USD'
  'SOL/USDC'     → 'SOL-USDC'

Round-trip verification:
  BTC/USD → BTC-USD → BTC/USD  ✓
  BTC/USDC → BTC-USDC → BTC/USDC  ✓
  LTC/USD → LTC-USD → LTC/USD  ✓
  LTC/USDC → LTC-USDC → LTC/USDC  ✓
  SOL/BTC → SOL-BTC → SOL/BTC  ✓
  SOL/USD → SOL-USD → SOL/USD  ✓
  SOL/USDC → SOL-USDC → SOL/USDC  ✓


In [9]:
# USD ≠ USDC verification (DEC-001)
# Fetch BTC-USD and BTC-USDC tickers — prices should differ
usd_usdc_pairs = [("BTC-USD", "BTC-USDC"), ("LTC-USD", "LTC-USDC"), ("SOL-USD", "SOL-USDC")]

print(f"{'Pair':12s} {'bidPx':>14s} {'askPx':>14s}")
print("-" * 45)
for usd_sym, usdc_sym in usd_usdc_pairs:
    for sym in [usd_sym, usdc_sym]:
        if sym not in found_pairs:
            print(f"{sym:12s} {'N/A (not listed)':>14s}")
            continue
        resp = httpx.get(f"{BASE_URL}/api/v5/market/ticker", params={"instId": sym})
        d = resp.json()["data"][0]
        print(f"{sym:12s} {d['bidPx']:>14s} {d['askPx']:>14s}")
        time.sleep(0.2)
    print()

Pair                  bidPx          askPx
---------------------------------------------
BTC-USD             68609.3        68609.4
BTC-USDC            68607.2        68607.3

LTC-USD               55.03          55.07
LTC-USDC              55.03          55.05

SOL-USD               85.85          85.86
SOL-USDC              85.86          85.87



## 3. Ticker Endpoint — Primary Data Source

OKX V5 ticker: `GET /api/v5/market/ticker?instId=BTC-USD`

Unlike Gemini (where ticker lacks bid/ask sizes), OKX ticker provides all
four fields needed for TopOfBook: `bidPx`, `bidSz`, `askPx`, `askSz`.

This means we can use the ticker endpoint directly — no need for order book.

In [10]:
# Fetch a single ticker and inspect the full response
resp = httpx.get(f"{BASE_URL}/api/v5/market/ticker", params={"instId": "BTC-USD"})
raw = resp.json()
print("Full BTC-USD ticker response:")
pprint(raw)

Full BTC-USD ticker response:
{'code': '0',
 'data': [{'askPx': '68609.4',
           'askSz': '1.46068671',
           'bidPx': '68609.3',
           'bidSz': '1.56258257',
           'high24h': '70095',
           'instId': 'BTC-USD',
           'instType': 'SPOT',
           'last': '68622.9',
           'lastSz': '0.0001',
           'low24h': '67267.9',
           'open24h': '68950.4',
           'sodUtc0': '68797.4',
           'sodUtc8': '67503.1',
           'ts': '1771282524012',
           'vol24h': '199.06147392',
           'volCcy24h': '13618729.723436285'}],
 'msg': ''}


In [11]:
# Verify all TopOfBook fields are present and non-empty
ticker = raw["data"][0]

REQUIRED_FIELDS = ["bidPx", "bidSz", "askPx", "askSz", "ts", "instId"]
print("Field verification:")
all_present = True
for field in REQUIRED_FIELDS:
    val = ticker.get(field)
    present = val is not None and val != ""
    status = "✓" if present else "✗ MISSING"
    print(f"  {field:10s} = {val!r:20s}  {status}")
    if not present:
        all_present = False

print(f"\nAll required fields present: {all_present}")

# Check timestamp format
ts = ticker["ts"]
print(f"\nTimestamp: {ts} (len={len(ts)})")
print(f"  As int: {int(ts)}")
print(f"  Digits: {len(ts)} → {'milliseconds' if len(ts) == 13 else 'UNEXPECTED'}")
print(f"  Approx date: {time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime(int(ts) / 1000))} UTC")

Field verification:
  bidPx      = '68609.3'             ✓
  bidSz      = '1.56258257'          ✓
  askPx      = '68609.4'             ✓
  askSz      = '1.46068671'          ✓
  ts         = '1771282524012'       ✓
  instId     = 'BTC-USD'             ✓

All required fields present: True

Timestamp: 1771282524012 (len=13)
  As int: 1771282524012
  Digits: 13 → milliseconds
  Approx date: 2026-02-16 22:55:24 UTC


In [12]:
# Fetch tickers for all available target pairs
print(
    f"{'Pair':12s} {'bidPx':>14s} {'bidSz':>14s} {'askPx':>14s} {'askSz':>14s} {'ts (last 6)':>12s}"
)
print("-" * 85)

pair_tickers = {}
for canonical, okx_sym in sorted(OKX_SYMBOL_MAP.items()):
    resp = httpx.get(f"{BASE_URL}/api/v5/market/ticker", params={"instId": okx_sym})
    body = resp.json()
    if body["code"] != "0" or not body["data"]:
        print(f"{canonical:12s} ERROR: code={body['code']} msg={body['msg']}")
        continue
    d = body["data"][0]
    pair_tickers[canonical] = d
    print(
        f"{canonical:12s} {d['bidPx']:>14s} {d['bidSz']:>14s} "
        f"{d['askPx']:>14s} {d['askSz']:>14s} {d['ts'][-6:]:>12s}"
    )
    time.sleep(0.2)

print(f"\nSuccessfully fetched {len(pair_tickers)}/{len(OKX_SYMBOL_MAP)} pairs")

Pair                  bidPx          bidSz          askPx          askSz  ts (last 6)
-------------------------------------------------------------------------------------
BTC/USD             68609.3     1.47502008        68609.4     1.46068671       523214
BTC/USDC            68607.2     3.50179831        68607.3     3.35219763       524604
LTC/USD               55.03      34.556621          55.07      28.435413       524604
LTC/USDC              55.03       18.17274          55.04        18.1686       525004
SOL/BTC           0.0012513         50.755      0.0012516        57.8557       525736
SOL/USD               85.86      58.244334          85.87         1.1773       525805
SOL/USDC              85.86       11.64714          85.87        5.82275       516604

Successfully fetched 7/7 pairs


## 4. Batch Tickers vs Single Ticker

OKX offers `GET /api/v5/market/tickers?instType=SPOT` which returns ALL spot
instruments in one call. Like Kraken's batch ticker, this could save us 7 API
calls per scan cycle (1 call vs 8).

Trade-off: batch returns hundreds of instruments but we only need 8.
Let's measure both approaches.

In [13]:
# Fetch batch tickers
t0 = time.monotonic()
resp = httpx.get(f"{BASE_URL}/api/v5/market/tickers", params={"instType": "SPOT"})
batch_elapsed_ms = (time.monotonic() - t0) * 1000
batch_body = resp.json()

print(f"Batch request: {batch_elapsed_ms:.0f}ms, {len(batch_body['data'])} instruments")

# Filter to our target pairs
target_set = set(found_pairs.keys())
batch_filtered = {d["instId"]: d for d in batch_body["data"] if d["instId"] in target_set}
print(f"After filtering: {len(batch_filtered)} target pairs")

# Show filtered results
print(f"\n{'instId':12s} {'bidPx':>14s} {'askPx':>14s}")
print("-" * 45)
for inst_id in sorted(batch_filtered.keys()):
    d = batch_filtered[inst_id]
    print(f"{inst_id:12s} {d['bidPx']:>14s} {d['askPx']:>14s}")

Batch request: 247ms, 736 instruments
After filtering: 7 target pairs

instId                bidPx          askPx
---------------------------------------------
BTC-USD             68609.3        68609.4
BTC-USDC            68607.2        68607.3
LTC-USD               55.03          55.07
LTC-USDC              55.03          55.04
SOL-BTC           0.0012514      0.0012516
SOL-USD               85.86          85.87
SOL-USDC              85.86          85.87


In [ ]:
# Compare timing: batch (1 call) vs per-pair (N calls)
available_pairs = list(found_pairs.keys())

# Time batch approach (3 runs, take median)
batch_times = []
for _ in range(3):
    t0 = time.monotonic()
    resp = httpx.get(f"{BASE_URL}/api/v5/market/tickers", params={"instType": "SPOT"})
    resp.json()  # force parse
    batch_times.append((time.monotonic() - t0) * 1000)
    time.sleep(0.3)

# Time per-pair approach (1 run)
t0 = time.monotonic()
for sym in available_pairs:
    resp = httpx.get(f"{BASE_URL}/api/v5/market/ticker", params={"instId": sym})
    resp.json()
    time.sleep(0.2)  # conservative rate limiting
per_pair_elapsed_ms = (time.monotonic() - t0) * 1000

n_pairs = len(available_pairs)
batch_median = sorted(batch_times)[len(batch_times) // 2]
print(f"Batch approach:    {batch_median:.0f}ms (median of 3 runs, 1 API call)")
print(
    f"Per-pair approach:  {per_pair_elapsed_ms:.0f}ms"
    f" ({n_pairs} API calls with 200ms delay)"
)
print(f"\nBatch is ~{per_pair_elapsed_ms / batch_median:.1f}x faster")
print(
    f"\nRecommendation:"
    f" {'BATCH' if batch_median < per_pair_elapsed_ms else 'PER-PAIR'} approach"
)

In [15]:
# Verify batch and single-ticker return consistent data
# (Prices may have shifted between calls, so just verify structure matches)
if batch_filtered and pair_tickers:
    sample_pair = list(batch_filtered.keys())[0]
    batch_item = batch_filtered[sample_pair]
    canonical_key = sample_pair.replace("-", "/")
    single_item = pair_tickers.get(canonical_key, {})

    print(f"Structure comparison for {sample_pair}:")
    print(f"  Batch keys:  {sorted(batch_item.keys())}")
    print(f"  Single keys: {sorted(single_item.keys())}")
    print(f"  Keys match:  {sorted(batch_item.keys()) == sorted(single_item.keys())}")
    print(f"\n  Batch  bidPx={batch_item['bidPx']}  askPx={batch_item['askPx']}")
    print(
        f"  Single bidPx={single_item.get('bidPx', 'N/A')}  askPx={single_item.get('askPx', 'N/A')}"
    )
    print("  (Values may differ due to market movement between calls — structure is what matters)")

Structure comparison for SOL-USDC:
  Batch keys:  ['askPx', 'askSz', 'bidPx', 'bidSz', 'high24h', 'instId', 'instType', 'last', 'lastSz', 'low24h', 'open24h', 'sodUtc0', 'sodUtc8', 'ts', 'vol24h', 'volCcy24h']
  Single keys: ['askPx', 'askSz', 'bidPx', 'bidSz', 'high24h', 'instId', 'instType', 'last', 'lastSz', 'low24h', 'open24h', 'sodUtc0', 'sodUtc8', 'ts', 'vol24h', 'volCcy24h']
  Keys match:  True

  Batch  bidPx=85.86  askPx=85.87
  Single bidPx=85.86  askPx=85.87
  (Values may differ due to market movement between calls — structure is what matters)


## 5. Parse into TopOfBook

Write a prototype parser that converts OKX V5 ticker response items into our
`TopOfBook` dataclass using `tob_from_raw()` — validation at the boundary (DEC-003).

Key differences from other parsers:
- Fields are `bidPx`/`bidSz`/`askPx`/`askSz` (camelCase, not nested)
- Timestamp is Unix ms as string — just `int(ts)`, no multiplication
- Response envelope: check `code == "0"` before accessing `data`

In [16]:
def parse_okx_ticker(data: dict, pair: str, ts_local_ms: int) -> TopOfBook:
    """Parse a single OKX V5 ticker data item into TopOfBook.

    Args:
        data: Single item from the response `data` array.
        pair: Canonical pair (e.g., "BTC/USD").
        ts_local_ms: Local timestamp in milliseconds.

    Returns:
        TopOfBook with validated Decimal fields.
    """
    return tob_from_raw(
        venue="okx",
        pair=pair,
        bid_px=data["bidPx"],
        bid_sz=data["bidSz"],
        ask_px=data["askPx"],
        ask_sz=data["askSz"],
        ts_local_ms=ts_local_ms,
        ts_exchange_ms=int(data["ts"]),
    )


print("parse_okx_ticker() defined.")

parse_okx_ticker() defined.


In [17]:
# Parse all available target pairs into TopOfBook
ts_now = int(time.time() * 1000)

print("Parsing live data into TopOfBook:")
print("-" * 80)

parsed_tobs = {}
for canonical, okx_sym in sorted(OKX_SYMBOL_MAP.items()):
    resp = httpx.get(f"{BASE_URL}/api/v5/market/ticker", params={"instId": okx_sym})
    body = resp.json()
    if body["code"] != "0" or not body["data"]:
        print(f"  {canonical:12s} SKIPPED (code={body['code']})")
        continue

    tob = parse_okx_ticker(body["data"][0], canonical, ts_now)
    parsed_tobs[canonical] = tob

    spread = tob.ask_px - tob.bid_px
    print(
        f"  {canonical:12s} bid={str(tob.bid_px):>14s} ask={str(tob.ask_px):>14s} "
        f" spread={spread}  venue={tob.venue}"
    )
    time.sleep(0.2)

print(f"\nParsed {len(parsed_tobs)}/{len(OKX_SYMBOL_MAP)} pairs into TopOfBook")

Parsing live data into TopOfBook:
--------------------------------------------------------------------------------
  BTC/USD      bid=       68609.3 ask=       68609.4  spread=0.1  venue=okx
  BTC/USDC     bid=       68607.2 ask=       68607.3  spread=0.1  venue=okx
  LTC/USD      bid=         55.02 ask=         55.05  spread=0.03  venue=okx
  LTC/USDC     bid=         55.03 ask=         55.04  spread=0.01  venue=okx
  SOL/BTC      bid=     0.0012508 ask=      0.001251  spread=2E-7  venue=okx
  SOL/USD      bid=          85.8 ask=         85.81  spread=0.01  venue=okx
  SOL/USDC     bid=         85.82 ask=         85.83  spread=0.01  venue=okx

Parsed 7/7 pairs into TopOfBook


In [18]:
# Verify Decimal types (LL-010: no floats for money)
if parsed_tobs:
    sample = list(parsed_tobs.values())[0]
    print(f"Sample TopOfBook ({sample.pair}):")
    print(f"  venue:          {sample.venue!r}")
    print(f"  pair:           {sample.pair!r}")
    print(f"  bid_px:         {sample.bid_px!r}  type={type(sample.bid_px).__name__}")
    print(f"  bid_sz:         {sample.bid_sz!r}  type={type(sample.bid_sz).__name__}")
    print(f"  ask_px:         {sample.ask_px!r}  type={type(sample.ask_px).__name__}")
    print(f"  ask_sz:         {sample.ask_sz!r}  type={type(sample.ask_sz).__name__}")
    print(f"  ts_local_ms:    {sample.ts_local_ms!r}")
    print(f"  ts_exchange_ms: {sample.ts_exchange_ms!r}")
    print(f"  frozen:         {sample.__dataclass_params__.frozen}")

Sample TopOfBook (BTC/USD):
  venue:          'okx'
  pair:           'BTC/USD'
  bid_px:         Decimal('68609.3')  type=Decimal
  bid_sz:         Decimal('1.59320011')  type=Decimal
  ask_px:         Decimal('68609.4')  type=Decimal
  ask_sz:         Decimal('1.46068671')  type=Decimal
  ts_local_ms:    1771282532246
  ts_exchange_ms: 1771282529916
  frozen:         True


## 6. Async httpx Pattern

Production connector will use `async with httpx.AsyncClient()` via
`BaseAsyncConnector._fetch_with_retry()`. Let's prototype both the batch
and per-pair async patterns.

In [19]:
# LL-064: Python 3.14 + nest_asyncio + anyio + httpx AsyncClient is broken.
# The weak-reference failure occurs inside httpcore's connection pool cleanup
# (AsyncShieldCancellation → anyio CancelScope → WeakKeyDictionary).
# This affects ALL async httpx usage in Jupyter on 3.14, regardless of how
# the coroutine is invoked (await, run_until_complete, etc.).
#
# Workaround: Use sync httpx to validate the data flow here.
# Production connector uses BaseAsyncConnector which runs outside Jupyter.

# --- Async code for production reference (DO NOT RUN in Jupyter 3.14) ---
#
# async def fetch_batch_tickers_async(
#     base_url: str, target_symbols: set[str]
# ) -> dict[str, dict]:
#     """Fetch all spot tickers in one call, filter to targets."""
#     async with httpx.AsyncClient(timeout=10.0) as client:
#         resp = await client.get(
#             f"{base_url}/api/v5/market/tickers",
#             params={"instType": "SPOT"},
#         )
#         body = resp.json()
#         if body["code"] != "0":
#             raise ValueError(f"OKX API error: code={body['code']} msg={body['msg']}")
#         return {
#             d["instId"]: d for d in body["data"] if d["instId"] in target_symbols
#         }

# --- Sync equivalent to validate the pattern ---
def fetch_batch_tickers_sync(base_url: str, target_symbols: set[str]) -> dict[str, dict]:
    """Sync version of batch fetch for Jupyter 3.14 testing."""
    resp = httpx.get(
        f"{base_url}/api/v5/market/tickers",
        params={"instType": "SPOT"},
        timeout=10.0,
    )
    body = resp.json()
    if body["code"] != "0":
        raise ValueError(f"OKX API error: code={body['code']} msg={body['msg']}")
    return {d["instId"]: d for d in body["data"] if d["instId"] in target_symbols}


target_syms = set(found_pairs.keys())
batch_result = fetch_batch_tickers_sync(BASE_URL, target_syms)
print(f"Batch fetch: {len(batch_result)} pairs")
for inst_id, d in sorted(batch_result.items()):
    print(f"  {inst_id}: bid={d['bidPx']} ask={d['askPx']}")

Batch fetch: 7 pairs
  BTC-USD: bid=68598 ask=68598.1
  BTC-USDC: bid=68607.2 ask=68607.3
  LTC-USD: bid=55.02 ask=55.05
  LTC-USDC: bid=55.03 ask=55.04
  SOL-BTC: bid=0.0012509 ask=0.001251
  SOL-USD: bid=85.8 ask=85.81
  SOL-USDC: bid=85.82 ask=85.83


In [20]:
# Sync equivalent of per-pair fetch (see LL-064 note above)
def fetch_per_pair_tickers_sync(
    base_url: str, symbols: list[str], delay_ms: int = 200
) -> dict[str, dict]:
    """Sync version of per-pair fetch for Jupyter 3.14 testing."""
    results = {}
    for sym in symbols:
        resp = httpx.get(
            f"{base_url}/api/v5/market/ticker",
            params={"instId": sym},
            timeout=10.0,
        )
        body = resp.json()
        if body["code"] == "0" and body["data"]:
            results[sym] = body["data"][0]
        else:
            print(f"  Warning: {sym} failed — code={body['code']}")
        if delay_ms > 0:
            time.sleep(delay_ms / 1000)
    return results


per_pair_result = fetch_per_pair_tickers_sync(BASE_URL, list(found_pairs.keys()))
print(f"Per-pair fetch: {len(per_pair_result)} pairs")
for inst_id, d in sorted(per_pair_result.items()):
    print(f"  {inst_id}: bid={d['bidPx']} ask={d['askPx']}")

print("\nNote: Production connector uses async httpx via BaseAsyncConnector.")
print("Async httpx cannot run in Jupyter on Python 3.14 (LL-064).")

Per-pair fetch: 7 pairs
  BTC-USD: bid=68598 ask=68598.1
  BTC-USDC: bid=68607.2 ask=68607.3
  LTC-USD: bid=55.02 ask=55.05
  LTC-USDC: bid=55.03 ask=55.04
  SOL-BTC: bid=0.0012509 ask=0.0012511
  SOL-USD: bid=85.8 ask=85.81
  SOL-USDC: bid=85.82 ask=85.83

Note: Production connector uses async httpx via BaseAsyncConnector.
Async httpx cannot run in Jupyter on Python 3.14 (LL-064).


## 7. Rate Limit Testing

OKX documents IP-based rate limits for public endpoints.
Error code `50011` = rate limit reached.

Let's test burst and sustained request patterns to find the safe interval.

In [21]:
# Burst test: send 20 rapid requests, check for rate limiting
BURST_COUNT = 20
burst_results = []

print(f"Burst test: {BURST_COUNT} rapid requests to /api/v5/market/ticker")
t0 = time.monotonic()

for i in range(BURST_COUNT):
    req_start = time.monotonic()
    resp = httpx.get(
        f"{BASE_URL}/api/v5/market/ticker",
        params={"instId": "BTC-USD"},
    )
    req_ms = (time.monotonic() - req_start) * 1000
    body = resp.json()
    burst_results.append(
        {
            "i": i,
            "status": resp.status_code,
            "code": body.get("code", "?"),
            "ms": req_ms,
        }
    )

total_ms = (time.monotonic() - t0) * 1000

# Check for any rate-limited responses
limited = [r for r in burst_results if r["code"] != "0" or r["status"] != 200]
print(f"  Total time: {total_ms:.0f}ms for {BURST_COUNT} requests")
print(f"  Avg per request: {total_ms / BURST_COUNT:.0f}ms")
print(f"  Rate-limited: {len(limited)} / {BURST_COUNT}")

if limited:
    print("\n  Rate-limited responses:")
    for r in limited:
        print(f"    Request #{r['i']}: HTTP {r['status']}, code={r['code']}")
else:
    print("  No rate limiting detected at burst speed.")

Burst test: 20 rapid requests to /api/v5/market/ticker
  Total time: 3922ms for 20 requests
  Avg per request: 196ms
  Rate-limited: 0 / 20
  No rate limiting detected at burst speed.


In [22]:
# Sustained test: 30 requests with different intervals
INTERVALS_MS = [50, 100, 200]

for interval_ms in INTERVALS_MS:
    count = 15
    errors = 0
    t0 = time.monotonic()
    for _ in range(count):
        resp = httpx.get(
            f"{BASE_URL}/api/v5/market/ticker",
            params={"instId": "BTC-USD"},
        )
        body = resp.json()
        if body.get("code") != "0" or resp.status_code != 200:
            errors += 1
        time.sleep(interval_ms / 1000)
    elapsed = (time.monotonic() - t0) * 1000
    print(f"  {interval_ms}ms interval: {count} requests in {elapsed:.0f}ms, errors={errors}")

print("\nRate limiter recommendation for production:")
print("  If using BATCH: interval barely matters (1 call/cycle)")
print("  If using PER-PAIR: 200ms interval should be safe")

  50ms interval: 15 requests in 3775ms, errors=0
  100ms interval: 15 requests in 4615ms, errors=0
  200ms interval: 15 requests in 6087ms, errors=0

Rate limiter recommendation for production:
  If using BATCH: interval barely matters (1 call/cycle)
  If using PER-PAIR: 200ms interval should be safe


## 8. Error Handling

OKX V5 API uses HTTP 200 for most errors — errors are indicated by a
non-zero `code` field in the JSON response. This is different from
Coinbase (HTTP 4xx) and Gemini (mixed HTTP codes + JSON errors).

In [23]:
# Error 1: Invalid instrument ID
resp = httpx.get(f"{BASE_URL}/api/v5/market/ticker", params={"instId": "FAKE-PAIR"})
print("Invalid instrument:")
print(f"  HTTP status: {resp.status_code}")
body = resp.json()
print(f"  code: {body['code']}")
print(f"  msg:  {body['msg']}")
print(f"  data: {body['data']}")

Invalid instrument:
  HTTP status: 200
  code: 51001
  msg:  Instrument ID, Instrument ID code, or Spread ID doesn't exist.
  data: []


In [24]:
# Error 2: Missing required parameter
resp = httpx.get(f"{BASE_URL}/api/v5/market/ticker")
print("Missing instId parameter:")
print(f"  HTTP status: {resp.status_code}")
body = resp.json()
print(f"  code: {body['code']}")
print(f"  msg:  {body['msg']}")
print(f"  data: {body['data']}")

Missing instId parameter:
  HTTP status: 400
  code: 50014
  msg:  Parameter instId can not be empty.
  data: []


In [25]:
# Error 3: Invalid instType
resp = httpx.get(f"{BASE_URL}/api/v5/market/tickers", params={"instType": "FAKETYPE"})
print("Invalid instType:")
print(f"  HTTP status: {resp.status_code}")
body = resp.json()
print(f"  code: {body['code']}")
print(f"  msg:  {body['msg']}")

Invalid instType:
  HTTP status: 400
  code: 51000
  msg:  Parameter instType error


In [26]:
# Error 4: Non-existent endpoint
resp = httpx.get(f"{BASE_URL}/api/v5/market/nonexistent")
print("Non-existent endpoint:")
print(f"  HTTP status: {resp.status_code}")
try:
    body = resp.json()
    print(f"  Response (JSON): {body}")
except Exception:
    print(f"  Response (text): {resp.text[:200]}")

Non-existent endpoint:
  HTTP status: 404
  Response (JSON): {'code': 404, 'msg': 'Not Found', 'error_code': '404', 'error_message': 'Not Found', 'detailMsg': '', 'data': {'timestamp': '2026-02-16T22:55:57.373+00:00', 'status': 404, 'error': 'Not Found', 'path': '/api/v5/market/nonexistent'}}


In [27]:
print("Error handling summary for production connector:")
print("")
print("  OKX V5 API error pattern:")
print("  - Most errors return HTTP 200 with non-zero 'code' in JSON body")
print("  - Success: code='0', data=[...]")
print("  - Error:   code='5xxxx', msg='...', data=[]")
print("")
print("  Production parser MUST:")
print("  1. Check HTTP status first (for network/server errors)")
print("  2. Parse JSON and check code == '0'")
print("  3. Verify data array is non-empty")
print("  4. Only then parse data[0] into TopOfBook")
print("")
print("  This is DIFFERENT from Coinbase (HTTP 4xx for errors)")
print("  and similar to Kraken ({'error': [...], 'result': {...}})")

Error handling summary for production connector:

  OKX V5 API error pattern:
  - Most errors return HTTP 200 with non-zero 'code' in JSON body
  - Success: code='0', data=[...]
  - Error:   code='5xxxx', msg='...', data=[]

  Production parser MUST:
  1. Check HTTP status first (for network/server errors)
  2. Parse JSON and check code == '0'
  3. Verify data array is non-empty
  4. Only then parse data[0] into TopOfBook

  This is DIFFERENT from Coinbase (HTTP 4xx for errors)
  and similar to Kraken ({'error': [...], 'result': {...}})


## 9. Symbol Details (Precision & Min Sizes)

Fetch instrument details for trading precision — needed for order sizing
in Phase 4 and for `TradingAccuracy` in `fee_schedules.json`.

Endpoint: `GET /api/v5/public/instruments?instType=SPOT&instId=BTC-USD`

In [28]:
# Fetch instrument details for all available target pairs
print(f"{'Pair':12s} {'tickSz':>12s} {'lotSz':>12s} {'minSz':>12s} {'State':>8s}")
print("-" * 65)

all_details = {}
for canonical, okx_sym in sorted(OKX_SYMBOL_MAP.items()):
    resp = httpx.get(
        f"{BASE_URL}/api/v5/public/instruments",
        params={"instType": "SPOT", "instId": okx_sym},
    )
    body = resp.json()
    if body["code"] != "0" or not body["data"]:
        print(f"{canonical:12s} ERROR: code={body['code']} msg={body['msg']}")
        continue

    details = body["data"][0]
    all_details[canonical] = details

    print(
        f"{canonical:12s} {details.get('tickSz', '?'):>12s} "
        f"{details.get('lotSz', '?'):>12s} "
        f"{details.get('minSz', '?'):>12s} "
        f"{details.get('state', '?'):>8s}"
    )
    time.sleep(0.2)

print(f"\nFetched details for {len(all_details)}/{len(OKX_SYMBOL_MAP)} pairs")

Pair               tickSz        lotSz        minSz    State
-----------------------------------------------------------------
BTC/USD               0.1   0.00000001       0.0001     live
BTC/USDC              0.1   0.00000001       0.0001     live
LTC/USD              0.01     0.000001         0.01     live
LTC/USDC             0.01      0.00001         0.01     live
SOL/BTC         0.0000001       0.0001        0.001     live
SOL/USD              0.01     0.000001         0.01     live
SOL/USDC             0.01      0.00001         0.01     live

Fetched details for 7/7 pairs


In [29]:
# Inspect one full instrument details response
if all_details:
    sample_key = sorted(all_details.keys())[0]
    print(f"Full instrument details for {sample_key}:")
    pprint(all_details[sample_key])

Full instrument details for BTC/USD:
{'alias': '',
 'auctionEndTime': '',
 'baseCcy': 'BTC',
 'category': '1',
 'contTdSwTime': '',
 'ctMult': '',
 'ctType': '',
 'ctVal': '',
 'ctValCcy': '',
 'expTime': '',
 'futureSettlement': False,
 'groupId': '12',
 'instCategory': '1',
 'instFamily': '',
 'instId': 'BTC-USD',
 'instIdCode': 187360,
 'instType': 'SPOT',
 'lever': '',
 'listTime': '1733454000000',
 'longPosRemainingQuota': '',
 'lotSz': '0.00000001',
 'maxIcebergSz': '9999999999.0000000000000000',
 'maxLmtAmt': '20000000',
 'maxLmtSz': '9999999999',
 'maxMktAmt': '120000',
 'maxMktSz': '1000000',
 'maxPlatOILmt': '',
 'maxStopSz': '1000000',
 'maxTriggerSz': '9999999999.0000000000000000',
 'maxTwapSz': '9999999999.0000000000000000',
 'minSz': '0.0001',
 'openType': 'fix_price',
 'optType': '',
 'posLmtAmt': '',
 'posLmtPct': '',
 'preMktSwTime': '',
 'quoteCcy': 'USD',
 'ruleType': 'normal',
 'settleCcy': '',
 'shortPosRemainingQuota': '',
 'state': 'live',
 'stk': '',
 'tickSz': 

## 10. Timestamp Format Deep Dive

OKX appears to use Unix milliseconds as strings in the `ts` field.
This is the simplest format of all our exchanges:
- Kraken: Unix seconds as float (multiply by 1000)
- Coinbase: ISO 8601 microseconds (parse + convert)
- Gemini: Unix seconds as integer string (multiply by 1000)
- OKX: Unix milliseconds as string (just `int(ts)` — no multiplication!)

In [30]:
# Verify timestamp format from ticker
resp = httpx.get(f"{BASE_URL}/api/v5/market/ticker", params={"instId": "BTC-USD"})
ticker_ts = resp.json()["data"][0]["ts"]

print("Ticker timestamp:")
print(f"  Raw value: {ticker_ts!r}")
print(f"  Type:      {type(ticker_ts).__name__}")
print(f"  Length:    {len(ticker_ts)} digits")
print(f"  As int:    {int(ticker_ts)}")
print(f"  As date:   {time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime(int(ticker_ts) / 1000))} UTC")

# Compare with server time
resp_time = httpx.get(f"{BASE_URL}/api/v5/public/time")
server_ts = resp_time.json()["data"][0]["ts"]

print("\nServer time:")
print(f"  Raw value: {server_ts!r}")
print(f"  Length:    {len(server_ts)} digits")
print(f"  As date:   {time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime(int(server_ts) / 1000))} UTC")

print("\nLocal time:")
local_ms = int(time.time() * 1000)
print(f"  Value:     {local_ms}")
print(f"  As date:   {time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime(local_ms / 1000))} UTC")

drift_ms = local_ms - int(server_ts)
print(f"\nClock drift: {drift_ms}ms (local - server)")

Ticker timestamp:
  Raw value: '1771282558310'
  Type:      str
  Length:    13 digits
  As int:    1771282558310
  As date:   2026-02-16 22:55:58 UTC

Server time:
  Raw value: '1771282560662'
  Length:    13 digits
  As date:   2026-02-16 22:56:00 UTC

Local time:
  Value:     1771282560692
  As date:   2026-02-16 22:56:00 UTC

Clock drift: 30ms (local - server)


In [31]:
# Check if the order book endpoint also has timestamps
resp = httpx.get(
    f"{BASE_URL}/api/v5/market/books",
    params={"instId": "BTC-USD", "sz": "1"},
)
book = resp.json()
print("Order book response (sz=1):")
pprint(book)

if book["code"] == "0" and book["data"]:
    book_data = book["data"][0]
    book_ts = book_data.get("ts", "NOT FOUND")
    print(f"\nOrder book timestamp: {book_ts!r}")
    print(
        f"  Same format as ticker: {len(str(book_ts)) == 13 if book_ts != 'NOT FOUND' else 'N/A'}"
    )

    # Note the order book bid/ask format for comparison
    bids = book_data.get("bids", [])
    asks = book_data.get("asks", [])
    if bids:
        print(f"\n  Bid format: {bids[0]}")
        print("  (Appears to be: [price, size, deprecated, num_orders])")
    if asks:
        print(f"  Ask format: {asks[0]}")

print("\nConversion for production connector:")
print("  ts_exchange_ms = int(data['ts'])  # Already milliseconds — no multiplication!")

Order book response (sz=1):
{'code': '0',
 'data': [{'asks': [['68613.3', '0.0002', '0', '2']],
           'bids': [['68613.2', '0.03549182', '0', '4']],
           'seqId': 4723454749,
           'ts': '1771282560806'}],
 'msg': ''}

Order book timestamp: '1771282560806'
  Same format as ticker: True

  Bid format: ['68613.2', '0.03549182', '0', '4']
  (Appears to be: [price, size, deprecated, num_orders])
  Ask format: ['68613.3', '0.0002', '0', '2']

Conversion for production connector:
  ts_exchange_ms = int(data['ts'])  # Already milliseconds — no multiplication!


## 11. Summary & Connector Design Notes

In [32]:
# Comparison table: all 4 exchanges
print("Exchange Comparison Table")
print("=" * 90)
print(f"{'Feature':<22s} {'Kraken':<16s} {'Coinbase':<16s} {'Gemini':<16s} {'OKX US':<16s}")
print("-" * 90)
rows = [
    ("Symbol format", "XXBTZUSD", "BTC-USD", "btcusd", "BTC-USD"),
    ("Batch endpoint", "YES (ticker)", "NO (per-pair)", "NO (per-pair)", "YES (tickers)"),
    ("BBO source", "Ticker", "product_book", "/v1/book", "Ticker"),
    ("Bid/ask sizes", "YES", "YES", "NO (use book)", "YES"),
    ("Timestamp format", "Unix s (float)", "ISO 8601 us", "Unix s (str)", "Unix ms (str)"),
    ("Ts conversion", "float()*1000", "parse ISO", "int()*1000", "int() only"),
    ("Rate limit", "~1 req/sec", "10 req/sec", "2 req/sec", "TBD above"),
    ("Rate limiter ms", "500ms", "100ms", "500ms", "TBD above"),
    ("SDK", "custom httpx", "custom httpx", "custom httpx", "custom httpx"),
    ("Exchange ts in", "Orderbook only", "All responses", "Orderbook only", "All responses"),
    ("Error format", "{error:[],res}", "HTTP 4xx", "Mixed", "HTTP 200+code"),
    ("Maker fee", "0.26%", "0.60%", "0.40%", "0.08%"),
    ("Taker fee", "0.26%", "0.60%", "0.40%", "0.10%"),
]
for label, *vals in rows:
    print(f"{label:<22s} {vals[0]:<16s} {vals[1]:<16s} {vals[2]:<16s} {vals[3]:<16s}")
print("=" * 90)

Exchange Comparison Table
Feature                Kraken           Coinbase         Gemini           OKX US          
------------------------------------------------------------------------------------------
Symbol format          XXBTZUSD         BTC-USD          btcusd           BTC-USD         
Batch endpoint         YES (ticker)     NO (per-pair)    NO (per-pair)    YES (tickers)   
BBO source             Ticker           product_book     /v1/book         Ticker          
Bid/ask sizes          YES              YES              NO (use book)    YES             
Timestamp format       Unix s (float)   ISO 8601 us      Unix s (str)     Unix ms (str)   
Ts conversion          float()*1000     parse ISO        int()*1000       int() only      
Rate limit             ~1 req/sec       10 req/sec       2 req/sec        TBD above       
Rate limiter ms        500ms            100ms            500ms            TBD above       
SDK                    custom httpx     custom httpx     custom 

In [33]:
print("Production Connector Design Notes")
print("=" * 60)
print()
print("1. ENDPOINT CHOICE: Ticker (not order book)")
print("   - Ticker provides all 4 TopOfBook fields (bidPx, bidSz, askPx, askSz)")
print("   - Unlike Gemini, no need to fall back to order book")
print()
print("2. FETCH STRATEGY: Batch vs Per-Pair")
print("   - Batch: 1 API call, filter client-side")
print("   - Per-pair: N API calls with rate limiting")
print("   - Recommend: batch (like Kraken) if timing results above confirm advantage")
print("   - If batch: override _fetch_tickers_per_pair with batch logic")
print("     OR implement fetch_tickers() directly (like KrakenClient)")
print()
print("3. RESPONSE PARSING:")
print("   - Check code=='0' (OKX returns HTTP 200 for API errors!)")
print("   - This is similar to Kraken's error array pattern")
print("   - Parse data[0] dict directly (flat structure, no nesting)")
print()
print("4. TIMESTAMP: Simplest of all exchanges")
print("   - int(data['ts']) — already milliseconds")
print("   - No multiplication, no ISO parsing")
print()
print("5. SYMBOL FORMAT: Same as Coinbase")
print("   - BTC-USD (dash-separated, uppercase)")
print("   - Canonical BTC/USD → OKX BTC-USD: just replace / with -")
print("   - But still use explicit OKX_SYMBOL_MAP (LL-001: verify, don't assume)")
print()
print("6. RATE LIMITING:")
print("   - If batch: minimal (1 call per cycle, well under any limit)")
print("   - If per-pair: 200ms interval (safe based on testing above)")
print()
print("7. ERROR HANDLING:")
print("   - HTTP errors: Let _fetch_with_retry() handle retries (5xx, 429)")
print("   - API errors: Check code != '0' after JSON parse")
print("   - Empty data: Log warning, skip pair")
print()
print(f"8. PAIRS AVAILABLE: {len(OKX_SYMBOL_MAP)}/{8}")
if missing_pairs:
    print(f"   MISSING: {missing_pairs}")
    print("   Production connector should only register available pairs")

Production Connector Design Notes

1. ENDPOINT CHOICE: Ticker (not order book)
   - Ticker provides all 4 TopOfBook fields (bidPx, bidSz, askPx, askSz)
   - Unlike Gemini, no need to fall back to order book

2. FETCH STRATEGY: Batch vs Per-Pair
   - Batch: 1 API call, filter client-side
   - Per-pair: N API calls with rate limiting
   - Recommend: batch (like Kraken) if timing results above confirm advantage
   - If batch: override _fetch_tickers_per_pair with batch logic
     OR implement fetch_tickers() directly (like KrakenClient)

3. RESPONSE PARSING:
   - Check code=='0' (OKX returns HTTP 200 for API errors!)
   - This is similar to Kraken's error array pattern
   - Parse data[0] dict directly (flat structure, no nesting)

4. TIMESTAMP: Simplest of all exchanges
   - int(data['ts']) — already milliseconds
   - No multiplication, no ISO parsing

5. SYMBOL FORMAT: Same as Coinbase
   - BTC-USD (dash-separated, uppercase)
   - Canonical BTC/USD → OKX BTC-USD: just replace / with -
  

In [ ]:
print("Refactor Candidates (per Coding Rule 10.9)")
print("=" * 60)
print()
print("1. create_connectors() type union:")
print("   - Currently: cast(type[KrakenClient]")
print("     | type[CoinbaseClient] | type[GeminiClient], ...)")
print("   - After OKX: cast(type[Kraken..]")
print("     | type[Coinbase..] | type[Gemini..]")
print("     | type[OkxClient], ...)")
print("   - 4 types is unwieldy — consider using")
print("     BaseAsyncConnector directly")
print("   - Possible fix: All connectors share")
print("     BaseAsyncConnector constructor (DEC-018)")
print("     so the cast may be unnecessary. Investigate.")
print()
print("2. Batch ticker pattern:")
print("   - Kraken: batch via /0/public/Ticker (all pairs in 1 call)")
print("   - OKX: batch via /api/v5/market/tickers?instType=SPOT")
print("     (all pairs in 1 call)")
print("   - 2 connectors now use batch pattern.")
print("     Consider shared helper?")
print("   - However: response formats differ significantly.")
print("     No refactor yet.")
print()
print("3. Error envelope parsing:")
print("   - Kraken: {'error': [], 'result': {...}}")
print("     — check error array")
print("   - OKX: {'code': '0', 'msg': '', 'data': [...]}")
print("     — check code string")
print("   - Similar pattern but different structure.")
print("     No shared helper.")
print()
print("4. Symbol map construction:")
print("   - OKX format is identical to Coinbase (dash-separated)")
print("   - But we still use explicit maps per LL-001")
print("     (verify, don't generate)")
print("   - create_translator() factory already handles this")
print("     (no new pattern)")

In [35]:
print("Lessons Learned (to add to docs/LESSONS_LEARNED.md)")
print("=" * 60)
print()
print("LL-070 (pending): OKX V5 API returns HTTP 200 for application errors")
print("  Non-zero 'code' field indicates failure, not HTTP status.")
print("  Rule: Always check the JSON 'code' field after parsing, not just HTTP status.")
print("  Pattern is similar to Kraken (check 'error' array) but different format.")
print()
print("LL-071 (pending): OKX timestamps are already in milliseconds")
print("  Unlike Kraken (seconds float) or Gemini (seconds string),")
print("  OKX 'ts' field is Unix milliseconds as string. Just int(ts).")
print("  Rule: Don't multiply OKX timestamps by 1000 — they're already ms.")
print()
print("LL-072 (pending): OKCoin shutdown — verify exchange status before building")
print("  OKCoin ceased operations May 2025 with no advance API deprecation.")
print("  Rule: Before building any new connector, web-search '<exchange> status'")
print("  to verify the exchange is still operational.")

Lessons Learned (to add to docs/LESSONS_LEARNED.md)

LL-070 (pending): OKX V5 API returns HTTP 200 for application errors
  Non-zero 'code' field indicates failure, not HTTP status.
  Rule: Always check the JSON 'code' field after parsing, not just HTTP status.
  Pattern is similar to Kraken (check 'error' array) but different format.

LL-071 (pending): OKX timestamps are already in milliseconds
  Unlike Kraken (seconds float) or Gemini (seconds string),
  OKX 'ts' field is Unix milliseconds as string. Just int(ts).
  Rule: Don't multiply OKX timestamps by 1000 — they're already ms.

LL-072 (pending): OKCoin shutdown — verify exchange status before building
  OKCoin ceased operations May 2025 with no advance API deprecation.
  Rule: Before building any new connector, web-search '<exchange> status'
  to verify the exchange is still operational.


In [36]:
# Generate fixture data for tests — save raw responses
# (Run this cell after verifying all outputs above)

FIXTURE_PAIRS = {}
# Pick 3 representative pairs: 1 USD, 1 USDC, 1 crypto-cross
fixture_candidates = [
    ("BTC/USD", "BTC-USD"),
    ("LTC/BTC", "LTC-BTC"),
    ("SOL/BTC", "SOL-BTC"),
]

print("Fixture responses for test data:\n")
for canonical, okx_sym in fixture_candidates:
    if okx_sym not in found_pairs:
        print(f"--- {canonical} ({okx_sym}) --- SKIPPED (not available)")
        continue

    resp = httpx.get(f"{BASE_URL}/api/v5/market/ticker", params={"instId": okx_sym})
    data = resp.json()
    FIXTURE_PAIRS[canonical] = data
    print(f"--- {canonical} ({okx_sym}) ---")
    print(json.dumps(data, indent=2))
    print()
    time.sleep(0.3)

# Also save an error response as fixture
resp = httpx.get(f"{BASE_URL}/api/v5/market/ticker", params={"instId": "FAKE-PAIR"})
print("--- Error response fixture ---")
print(json.dumps(resp.json(), indent=2))

print("\n" + "=" * 60)
print("Copy these responses to tests/fixtures/ when building the production connector.")
print("Naming convention: okx_ticker_btc_usd.json, okx_ticker_ltc_btc.json, etc.")
print("Also save: okx_error_invalid_instrument.json")

Fixture responses for test data:

--- BTC/USD (BTC-USD) ---
{
  "code": "0",
  "msg": "",
  "data": [
    {
      "instType": "SPOT",
      "instId": "BTC-USD",
      "last": "68587.8",
      "lastSz": "0.01166612",
      "askPx": "68613.3",
      "askSz": "2.91465713",
      "bidPx": "68613.2",
      "bidSz": "1.53423281",
      "open24h": "68942.8",
      "high24h": "70095",
      "low24h": "67267.9",
      "volCcy24h": "13610073.083004305",
      "vol24h": "198.935942",
      "ts": "1771282560420",
      "sodUtc0": "68797.4",
      "sodUtc8": "67503.1"
    }
  ]
}

--- LTC/BTC (LTC-BTC) --- SKIPPED (not available)
--- SOL/BTC (SOL-BTC) ---
{
  "code": "0",
  "msg": "",
  "data": [
    {
      "instType": "SPOT",
      "instId": "SOL-BTC",
      "last": "0.0012511",
      "lastSz": "0.0264",
      "askPx": "0.0012507",
      "askSz": "8.8097",
      "bidPx": "0.0012506",
      "bidSz": "119.8066",
      "open24h": "0.0012531",
      "high24h": "0.001257",
      "low24h": "0.0012235",